# Bab 8: Mempelajari Sinyal dan Mengabaikan Kebisingan: Pengantar Regularisasi dan Batching
*(Learning Signal and Ignoring Noise: Intro to Regularization and Batching)*

## Eksplorasi Teoritis Mendalam (Theoretical Deep-Dive)

Bab ini mengatasi dua masalah terbesar dalam melatih jaringan saraf yang dalam (Deep Neural Networks): **Overfitting** (model menghafal data tetapi gagal di dunia nyata) dan **Kecepatan Pelatihan** (model terlalu lambat karena memproses data satu per satu). Andrew Trask memperkenalkan dua solusi standar industri untuk masalah ini: *Dropout* dan *Mini-Batch Stochastic Gradient Descent (SGD)*.

### 1. Sinyal vs. Kebisingan (Signal vs. Noise)
Setiap *dataset* di dunia nyata memiliki dua hal:
*   **Signal (Sinyal):** Pola inti yang benar-benar menentukan hasil (misalnya, awan mendung = hujan).
*   **Noise (Kebisingan/Noise):** Informasi acak yang kebetulan ada tetapi tidak relevan (misalnya, saat hujan turun, kebetulan ada kucing lewat).
Jaringan saraf yang terlalu kompleks memiliki terlalu banyak kapasitas memori (bobot yang terlalu banyak). Akibatnya, ia tidak hanya mempelajari "Sinyal", tetapi juga menghafal "Noise". Saat model diuji di dunia nyata, model gagal karena berfokus pada noise yang salah. Ini disebut **Overfitting**.

### 2. Regularisasi: Teknik Dropout
Untuk mencegah *Overfitting*, kita menggunakan teknik Regularisasi yang paling terkenal: **Dropout**.
*   **Mekanisme:** Selama proses pelatihan (Forward Propagation), kita secara acak **mematikan** (mengubah nilainya menjadi 0) persentase tertentu dari *hidden node* (misalnya 50% node dimatikan).
*   **Mengapa Ini Berhasil?** Jika setengah dari otak mesin dimatikan secara acak pada setiap iterasi, jaringan saraf tidak bisa mengandalkan hanya pada 1 atau 2 node "pintar" untuk menghafal data (noise). Algoritma terpaksa harus mendistribusikan pembelajarannya ke seluruh node yang ada, mencari pola "Sinyal" yang paling kuat dan persisten. Ini menghasilkan model yang jauh lebih tangguh (robust) dan mampu melakukan generalisasi pada data baru.

### 3. Mini-Batch Stochastic Gradient Descent (SGD)
Sejauh ini, kita melatih mesin dengan memasukkan 1 baris data, mengupdate bobot, memasukkan baris ke-2, mengupdate bobot lagi, dst. Jika ada 1 juta data, mesin harus melakukan 1 juta update per *epoch* (putaran). Ini sangat lambat.
*   **Batching:** Alih-alih 1 data per update, kita mengambil sebuah *batch* (kelompok) kecil data—misalnya 100 baris sekaligus.
*   Kita melakukan *forward propagation* untuk ke-100 data tersebut secara bersamaan menggunakan matriks, merata-ratakan *error*-nya, dan hanya melakukan 1 kali *update* bobot untuk 100 data tersebut.
*   **Keuntungan:** Proses ini 100x lebih cepat karena GPU sangat ahli dalam menghitung matriks besar secara paralel. Selain itu, merata-ratakan *error* dari 100 data membuat arah *Gradient Descent* menjadi lebih stabil (mulus) dan tidak melompat-lompat akibat 1 data yang anomali.

In [ ]:
import numpy as np

# =====================================================================
# REPRODUKSI KODE BAB 8: DROPOUT & MINI-BATCHING
# Melatih model menggunakan Batch 10 data sekaligus dan Regularisasi Dropout
# =====================================================================

np.random.seed(1)

# Fungsi Aktivasi & Turunannya
def relu(x):
    return (x > 0) * x

def relu2deriv(output):
    return output > 0

# 1. Mensimulasikan Dataset yang Cukup Besar
# Kita buat 100 baris observasi, masing-masing punya 3 input (fitur)
# Label kebenaran (y) ditentukan oleh logika sederhana untuk dipecahkan mesin
X = np.random.randint(2, size=(100, 3))
y = np.array([[row[0]] for row in X]) # Target = 1 jika kolom pertama = 1

# 2. Hyperparameters
alpha = 0.1          # Learning Rate
hidden_size = 10     # Jumlah Hidden Nodes (Lebih banyak dari bab 6)
dropout_rate = 0.5   # 50% node akan dimatikan secara acak setiap saat
batch_size = 10      # MINI-BATCH: Memproses 10 data sekaligus sebelum update bobot

# Inisialisasi Matriks Bobot
weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1

print("Memulai Pelatihan dengan Mini-Batch SGD dan Dropout...")
print("-" * 60)

# 3. Looping Pelatihan (Training Loop)
for iteration in range(50):
    layer_2_error = 0

    # Looping ini melompat setiap 10 indeks (Batching)
    for i in range(0, len(X), batch_size):

        # Mengambil 10 baris data sekaligus (MINI-BATCH)
        layer_0 = X[i:i+batch_size]
        layer_2_target = y[i:i+batch_size]

        # FORWARD PROPAGATION
        layer_1 = relu(np.dot(layer_0, weights_0_1))

        # =======================================
        # DROPOUT MASKING (REGULARIZATION)
        # =======================================
        # Membuat filter acak berisi 0 dan 1 (50% kemungkinan 0)
        dropout_mask = np.random.randint(2, size=layer_1.shape)

        # Mematikan node secara acak dengan mengalikannya dengan mask
        # Dikali 2 untuk menjaga skala total volume sinyal yang maju ke layer 2
        layer_1 *= dropout_mask * 2

        layer_2 = np.dot(layer_1, weights_1_2)

        # BACKPROPAGATION
        layer_2_delta = layer_2 - layer_2_target
        layer_2_error += np.sum(layer_2_delta ** 2)

        # Menarik mundur kesalahan ke Layer 1
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)

        # PENTING: Node yang mati saat Forward (Dropout),
        # juga harus dimatikan / tidak boleh diupdate saat Backward (Backprop)
        layer_1_delta *= dropout_mask

        # UPDATE BOBOT (1 kali update untuk 10 data observasi sekaligus)
        weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 -= alpha * layer_0.T.dot(layer_1_delta)

    if iteration % 10 == 9:
        print(f"Iterasi: {iteration+1:<3} | Total Batch Error: {layer_2_error:.4f}")

print("\nPelatihan selesai! Dropout berhasil mencegah overfitting, dan Batching mempercepat proses.")

### Penjelasan Eksekusi Kode:

1.  **Mini-Batching (`for i in range(0, len(X), batch_size):`)**: Berbeda dengan Bab 6 di mana `layer_0` berisi 1 baris observasi `[1, 0, 1]`, di sini `layer_0` adalah matriks berisi **10 baris observasi sekaligus**. Operasi `np.dot()` akan menghitung prediksi untuk ke-10 baris data tersebut secara paralel, menghasilkan `layer_2` yang berisi 10 tebakan sekaligus. Ini jauh lebih efisien untuk prosesor.
2.  **`dropout_mask = np.random.randint(2, size=layer_1.shape)`**: Baris ini menghasilkan matriks (dengan ukuran yang persis sama dengan ukuran matriks *hidden layer*) yang isinya hanya angka `0` dan `1` secara acak.
3.  **Mempertahankan Skala Volume (`layer_1 *= dropout_mask * 2`)**: Ketika kita mematikan setengah dari *hidden node* (mengalikannya dengan `0`), volume sinyal yang sampai ke lapisan akhir (`layer_2`) akan terpotong setengahnya. Agar prediksi `layer_2` tidak rusak akibat kurangnya tenaga sinyal, kita harus mengalikan sisa *node* yang masih hidup dengan `2` (atau `1 / (1 - 0.5)`).
4.  **`layer_1_delta *= dropout_mask`**: Aturan emas dari *Dropout*: Jika sebuah neuron sedang "tidur" (dimatikan) saat membuat prediksi, maka ia tidak boleh disalahkan atau dipuji saat evaluasi kesalahan (*Backpropagation*). Oleh karena itu, kita terapkan masker (mask) yang sama pada nilai *delta*-nya untuk memastikan bobotnya tidak diubah pada iterasi tersebut.